# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset package (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the `mlcroissant` library.

### Dataset Source
The dataset is defined via a Croissant schema JSON-LD file accessible here:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

_All entities (record sets, fields, columns, etc.) are referenced by their `@id` as per Croissant convention._

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset schema and metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Set the URL for the Croissant schema file
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Display basic metadata info (access via property, not dict subscript)
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review and display available record sets, their fields, and associated `@id` values.

In [ ]:
# List available RecordSets (by their @id), their fields, and columns
print("Available Record Sets and Fields:\n")
for record_set in metadata.record_sets:
    print(f"Record Set: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    - {field.id} (name='{field.name}', type={field.data_type})")
    print()

## 3. Data Extraction
Load records from one or multiple record sets into pandas DataFrames. You can reference each by its `@id` from the overview.

For this dataset, we'll process all available record sets.

In [ ]:
# Prepare a list of record set IDs (@id)
record_set_ids = [r.id for r in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading data for Record Set: {record_set_id}")
    recs = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(recs)
    dataframes[record_set_id] = df
    print(f"Columns: {list(df.columns)}\nShape: {df.shape}\n")

# For demo: pick the first record set to show data
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Sample rows from {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data wrangling: filtering, normalization, and grouping.

_Reminder: For all variables below, use the actual `@id` for columns (see the printed list above)._

In [ ]:
# Let's demonstrate with the main record set (first in the list)
df = dataframes[main_record_set_id].copy()

# Pick a numeric field: choose the first float/integer field from the schema fields
main_record_set_fields = [rs for rs in metadata.record_sets if rs.id == main_record_set_id][0].fields
import numpy as np
numeric_field_id = None
for field in main_record_set_fields:
    if field.data_type in ['Number', 'Float', 'Integer']:
        numeric_field_id = field.id
        break

if numeric_field_id is not None and numeric_field_id in df.columns:
    print(f"Using numeric field '@id': {numeric_field_id}")
    # Filter by threshold
    threshold = 10
    # Some columns may have non-numeric strings: try coercion
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered to {len(filtered_df)} records with {numeric_field_id} > {threshold}.")
    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    display(filtered_df[[numeric_field_id, norm_col]].head())
    # Group by a possible categorical field (choose a string/text field)
    group_field_id = None
    for field in main_record_set_fields:
        if field.data_type in ['Text', 'String'] and field.id in filtered_df.columns:
            group_field_id = field.id
            break
    if group_field_id:
        print(f"Grouping by field '@id': {group_field_id}")
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped.head())
else:
    print("No numeric field available for EDA in this record set.")

## 5. Visualization
Visualize distributions for a numeric field, grouping if possible.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot by a group field (if one is available)
    if group_field_id and group_field_id in df and df[group_field_id].nunique() < 20:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion
In this notebook, you loaded the Mass Spectrometry EV Proteomics dataset using `mlcroissant`, performed schema-driven exploration (by `@id`), filtered and normalized a numeric field, and visualized sample distributions. This process provides a reproducible and FAIR analysis pattern for Croissant-published datasets.

_For more advanced workflows, consider selecting different record sets or fields, computing correlations, or integrating with downstream ML pipelines._